In [1]:
!pip install -q torch transformers datasets pillow pandas scikit-learn tqdm

In [2]:
import os
import json
import torch
import torch.nn as nn
import torch.nn.functional as F
from tqdm.auto import tqdm
from torch.utils.data import Dataset, DataLoader
from datasets import load_dataset
from transformers import CLIPModel, CLIPImageProcessor

In [3]:
MODEL_NAME = "openai/clip-vit-large-patch14"
DATASET_NAME = "ashraq/fashion-product-images-small"

TASKS = ["gender", "masterCategory", "subCategory", "articleType", "baseColour", "season", "usage"]

In [4]:
OUTPUT_DIR = "artifacts/frozen_clip_multitask"
SPLIT = "train"
MAX_SAMPLES = None
VAL_RATIO = 0.1
SEED = 42
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
os.makedirs(OUTPUT_DIR, exist_ok=True)
torch.manual_seed(SEED)
DEVICE

'cuda'

In [5]:
BATCH_SIZE = 32
EPOCHS = 2
LR = 1e-3
WEIGHT_DECAY = 1e-4
VAL_RATIO = 0.1
HIDDEN_DIM = 512
DROPOUT = 0.2
NUM_WORKERS = 0

In [6]:
raw_dataset = load_dataset(DATASET_NAME, split=SPLIT)

raw_dataset

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


README.md:   0%|          | 0.00/867 [00:00<?, ?B/s]

data/train-00000-of-00002-6cff4c59f91661(…):   0%|          | 0.00/136M [00:00<?, ?B/s]

data/train-00001-of-00002-bb459e5ac5f01e(…):   0%|          | 0.00/135M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/44072 [00:00<?, ? examples/s]

Dataset({
    features: ['id', 'gender', 'masterCategory', 'subCategory', 'articleType', 'baseColour', 'season', 'year', 'usage', 'productDisplayName', 'image'],
    num_rows: 44072
})

In [7]:
raw_dataset.column_names

['id',
 'gender',
 'masterCategory',
 'subCategory',
 'articleType',
 'baseColour',
 'season',
 'year',
 'usage',
 'productDisplayName',
 'image']

In [8]:
missing_columns = [task for task in TASKS if task not in raw_dataset.column_names]

if "image" not in raw_dataset.column_names:
    raise ValueError(f"Dataset must contain an image column. Found columns: {raw_dataset.column_names}")
if missing_columns:
    raise ValueError(f"Missing task columns: {missing_columns}. Found columns: {raw_dataset.column_names}")

raw_dataset = raw_dataset.filter(lambda row: all(row[task] is not None and str(row[task]).strip() != "" for task in TASKS))

raw_dataset = raw_dataset.shuffle(seed=SEED)
if MAX_SAMPLES is not None:
    raw_dataset = raw_dataset.select(range(min(MAX_SAMPLES, len(raw_dataset))))

split_dataset = raw_dataset.train_test_split(test_size=VAL_RATIO, seed=SEED)
train_hf_dataset = split_dataset["train"]
val_hf_dataset = split_dataset["test"]

len(train_hf_dataset), len(val_hf_dataset)

Filter:   0%|          | 0/44072 [00:00<?, ? examples/s]

(39664, 4408)

In [9]:
def build_label_maps(dataset, tasks):
    label2id = {}
    id2label = {}
    for task in tasks:
        labels = sorted(set(str(value).strip() for value in dataset[task]))
        
        label2id[task] = {label: idx for idx, label in enumerate(labels)}
        
        id2label[task] = {str(idx): label for label, idx in label2id[task].items()}
        
    return label2id, id2label


label2id, id2label = build_label_maps(raw_dataset, TASKS)
task_num_classes = {task: len(label2id[task]) for task in TASKS}

with open(os.path.join(OUTPUT_DIR, "label2id.json"), "w", encoding="utf-8") as f:
    json.dump(label2id, f, indent=2, ensure_ascii=False)

with open(os.path.join(OUTPUT_DIR, "id2label.json"), "w", encoding="utf-8") as f:
    json.dump(id2label, f, indent=2, ensure_ascii=False)

task_num_classes

{'gender': 5,
 'masterCategory': 7,
 'subCategory': 45,
 'articleType': 141,
 'baseColour': 46,
 'season': 4,
 'usage': 8}

In [10]:
processor = CLIPImageProcessor.from_pretrained(MODEL_NAME)

preprocessor_config.json:   0%|          | 0.00/316 [00:00<?, ?B/s]

In [11]:
class HFFashionMultiTaskDataset(Dataset):
    def __init__(self, hf_dataset, processor, tasks, label2id):
        self.dataset = hf_dataset
        self.processor = processor
        self.tasks = tasks
        self.label2id = label2id

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        item = self.dataset[idx]
        image = item["image"].convert("RGB")
        inputs = self.processor(images=image, return_tensors="pt")
        
        labels = {
            task: torch.tensor(self.label2id[task][str(item[task]).strip()], dtype=torch.long)
            for task in self.tasks
        }
        
        return {
            "pixel_values": inputs["pixel_values"].squeeze(0),
            "labels": labels
        }


def multitask_collate_fn(batch):
    pixel_values = torch.stack([item["pixel_values"] for item in batch])
    labels = {
        task: torch.stack([item["labels"][task] for item in batch])
        for task in batch[0]["labels"]
    }
    return {
        "pixel_values": pixel_values,
        "labels": labels
    }

In [12]:
train_dataset = HFFashionMultiTaskDataset(train_hf_dataset, processor, TASKS, label2id)
val_dataset = HFFashionMultiTaskDataset(val_hf_dataset, processor, TASKS, label2id)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    collate_fn=multitask_collate_fn
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    collate_fn=multitask_collate_fn
)

batch = next(iter(train_loader))
batch["pixel_values"].shape, {k: v.shape for k, v in batch["labels"].items()}

(torch.Size([32, 3, 224, 224]),
 {'gender': torch.Size([32]),
  'masterCategory': torch.Size([32]),
  'subCategory': torch.Size([32]),
  'articleType': torch.Size([32]),
  'baseColour': torch.Size([32]),
  'season': torch.Size([32]),
  'usage': torch.Size([32])})

In [13]:
class ClassificationHead(nn.Module):
    def __init__(self, embedding_dim, num_classes, hidden_dim=512, dropout=0.2):
        super().__init__()
        self.net = nn.Sequential(
            nn.LayerNorm(embedding_dim),
            nn.Linear(embedding_dim, hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, num_classes)
        )
    
    def forward(self, x):
        return self.net(x)
        

In [14]:
class FrozenCLIPMultiTaskClassifier(nn.Module):
    def __init__(self, model_name, task_num_classes, hidden_dim=512, dropout=0.2):
        super().__init__()
        self.clip = CLIPModel.from_pretrained(model_name)
        
        for param in self.clip.parameters():
            param.requires_grad = False
        
        self.clip.eval()
        
        embedding_dim = self.clip.config.projection_dim
        self.heads = nn.ModuleDict({
            task_name : ClassificationHead(embedding_dim, num_classes, hidden_dim, dropout) for task_name, num_classes in task_num_classes.items()
        })
        
    def train(self, mode=True):
        super().train(mode)
        self.clip.eval()
        return self
    
    def forward(self, pixel_values):
        with torch.no_grad():
            image_features = self.clip.get_image_features(pixel_values=pixel_values).pooler_output
            
        image_features = F.normalize(image_features, dim=-1)
        
        return {task_name: head(image_features) for task_name, head in self.heads.items()}

In [15]:
model = FrozenCLIPMultiTaskClassifier(
    model_name=MODEL_NAME,
    task_num_classes=task_num_classes,
    hidden_dim=512,
    dropout=0.1
).to(DEVICE)

trainable_params = [param for param in model.parameters() if param.requires_grad]
optimizer = torch.optim.AdamW(trainable_params, lr=1e-3, weight_decay=1e-4)
criterions = {task: nn.CrossEntropyLoss() for task in TASKS}
task_weights = {task: 1.0 for task in TASKS}

config.json:   0%|          | 0.00/4.52k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.71G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/590 [00:00<?, ?it/s]

In [16]:
total_params = sum(param.numel() for param in model.parameters())
trainable_count = sum(param.numel() for param in model.parameters() if param.requires_grad)

{
    "total_params": total_params,
    "trainable_params": trainable_count,
    "frozen_params": total_params - trainable_count
}

{'total_params': 430514689,
 'trainable_params': 2898176,
 'frozen_params': 427616513}

In [17]:
def compute_loss(outputs, labels, criterions, task_weights):
    loss = 0.0
    for task_name, logits in outputs.items():
        loss = loss + task_weights[task_name] * criterions[task_name](logits, labels[task_name])
    return loss


def compute_accuracy(outputs, labels):
    result = {}
    for task_name, logits in outputs.items():
        preds = torch.argmax(logits, dim=1)
        correct = (preds == labels[task_name]).sum().item()
        total = labels[task_name].numel()
        result[task_name] = correct / total
    return result


def train_one_epoch(model, loader, optimizer, criterions, task_weights, device):
    model.train()
    total_loss = 0.0
    total_acc = {task: 0.0 for task in model.heads.keys()}

    for batch in tqdm(loader, desc="Training"):
        pixel_values = batch["pixel_values"].to(device)
        labels = {task: value.to(device) for task, value in batch["labels"].items()}

        optimizer.zero_grad()
        outputs = model(pixel_values)
        loss = compute_loss(outputs, labels, criterions, task_weights)
        loss.backward()
        optimizer.step()

        acc = compute_accuracy(outputs, labels)
        total_loss += loss.item()

        for task in total_acc:
            total_acc[task] += acc[task]

    total_loss /= len(loader)

    for task in total_acc:
        total_acc[task] /= len(loader)

    return total_loss, total_acc


@torch.no_grad()
def evaluate(model, loader, criterions, task_weights, device):
    model.eval()
    total_loss = 0.0
    total_acc = {task: 0.0 for task in model.heads.keys()}

    for batch in tqdm(loader, desc="Validation"):
        pixel_values = batch["pixel_values"].to(device)
        labels = {task: value.to(device) for task, value in batch["labels"].items()}

        outputs = model(pixel_values)
        loss = compute_loss(outputs, labels, criterions, task_weights)
        acc = compute_accuracy(outputs, labels)
        total_loss += loss.item()

        for task in total_acc:
            total_acc[task] += acc[task]

    total_loss /= len(loader)

    for task in total_acc:
        total_acc[task] /= len(loader)

    return total_loss, total_acc

In [ ]:
best_val_loss = float("inf")
history = []

for epoch in range(EPOCHS):
    train_loss, train_acc = train_one_epoch(model, train_loader, optimizer, criterions, task_weights, DEVICE)
    val_loss, val_acc = evaluate(model, val_loader, criterions, task_weights, DEVICE)

    record = {
        "epoch": epoch + 1,
        "train_loss": train_loss,
        "val_loss": val_loss,
        "train_acc": train_acc,
        "val_acc": val_acc
    }

    history.append(record)
    print(json.dumps(record, indent=2))

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        checkpoint = {
            "model_name": MODEL_NAME,
            "task_num_classes": task_num_classes,
            "heads_state_dict": model.heads.state_dict(),
            "label2id": label2id,
            "id2label": id2label,
            "tasks": TASKS,
            "hidden_dim": HIDDEN_DIM,
            "dropout": DROPOUT
        }
        torch.save(checkpoint, os.path.join(OUTPUT_DIR, "best_heads.pt"))

with open(os.path.join(OUTPUT_DIR, "history.json"), "w", encoding="utf-8") as f:
    json.dump(history, f, indent=2, ensure_ascii=False)

Training:   0%|          | 0/1240 [00:00<?, ?it/s]